<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Practica/AgenticAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## Чем VS Code отличается от Colab (важно понять сразу)

В Colab вы пишете код в **ячейках** и нажимаете «Play» рядом с ячейкой. В VS Code всё иначе:

- **Код хранится в файлах** (например, `chat.py`), а не в ячейках.
- **Запуск происходит через терминал** — текстовое окно внизу редактора, куда вы вводите команды (как в командной строке Windows).
- **Проект — это папка.** Вы открываете в VS Code целую папку, и все файлы внутри неё становятся частью проекта.

Если это понять, дальше всё будет логично.

---

## Шаг 1. Установите Python (если его ещё нет)

Colab запускает Python «в облаке», а VS Code запускает его **на вашем компьютере**. Поэтому Python нужно установить локально.

1. Откройте [python.org/downloads](https://www.python.org/downloads/) и скачайте последнюю стабильную версию для Windows.
2. Запустите установщик. **Обязательно поставьте галочку «Add Python to PATH»** внизу первого экрана — это критически важно.
3. Нажмите «Install Now» и дождитесь завершения.
4. Проверьте установку. Откройте **PowerShell** (нажмите `Win`, начните вводить `PowerShell` и откройте его) и введите:
   ```powershell
   python --version
   ```
   Если вы увидели номер версии (например, `Python 3.12.x`) — всё хорошо. Если нет — переустановите Python, убедившись, что галочка «Add to PATH» стоит.

> **Альтернатива:** если у вас Windows 10/11, Python можно установить одной командой в PowerShell: `winget install Python.Python.3.12`. После установки закройте и снова откройте PowerShell.

---

## Шаг 2. Установите VS Code

1. Откройте [code.visualstudio.com](https://code.visualstudio.com/) и скачайте установщик для Windows.
2. Запустите его и установите как обычную программу (можно оставить все настройки по умолчанию).
3. После установки откройте VS Code.

---

## Шаг 3. Установите расширение Python в VS Code

Расширение добавляет подсветку синтаксиса, подсказки и кнопку запуска файлов.

1. В VS Code нажмите `Ctrl+Shift+X` (откроется панель Extensions).
2. В поиске введите `Python`.
3. Найдите расширение **Python** от Microsoft (у него значок с синей змейкой) и нажмите **Install**.

После установки VS Code сам найдёт ваш Python.

---

## Шаг 4. Установите Ollama (движок для моделей)

Ollama — это программа, которая запускает большие языковые модели локально и предоставляет к ним доступ через простой API.

1. Откройте **PowerShell** (можно прямо в VS Code, но проще в отдельном окне Windows).
2. Введите команду:
   ```powershell
   winget install Ollama.Ollama
   ```
   Это официальный способ установки для Windows через встроенный менеджер пакетов.
3. Дождитесь завершения. После установки Ollama **автоматически запустится в фоне** — в системном трее (рядом с часами) появится её значок.
4. **Закройте и снова откройте PowerShell** (или перезапустите VS Code), чтобы система «увидела» новую команду. Проверьте:
   ```powershell
   ollama --version
   ```
   Если вывелась версия — установка успешна.

---

## Шаг 5. Скачайте лёгкую модель

Модель — это «мозг», который будет отвечать на ваши вопросы. Начнём с **Phi-3 Mini** — она маленькая (около 2 ГБ), быстро работает и понимает русский.

В том же PowerShell выполните:
```powershell
ollama pull phi3:mini
```
Дождитесь окончания загрузки. Проверить, что модель появилась, можно командой:
```powershell
ollama list
```

---

## Шаг 6. Создайте папку для проекта

В VS Code проект — это папка на вашем компьютере.

1. Откройте «Проводник» Windows (значок папки на панели задач).
2. Перейдите, например, в `C:\Users\ВашеИмя\Documents`.
3. Создайте новую папку с именем `my_ai_chat` (без пробелов и русских букв — так надёжнее).

---

## Шаг 7. Откройте папку в VS Code

1. В VS Code в верхнем меню выберите **File → Open Folder…** (Файл → Открыть папку).
2. Найдите созданную папку `my_ai_chat` и нажмите «Выбрать папку».
3. Если VS Code спросит, доверяете ли вы авторам файлов, нажмите **Yes, I trust the authors**.

Теперь в левой панели (Explorer) отображается содержимое папки — пока пустое.

---

## Шаг 8. Создайте файл Python и напишите код

1. В левой панели наведите курсор на имя папки `MY_AI_CHAT` — появятся иконки.
2. Нажмите иконку **New File** (лист с плюсом).
3. Введите имя файла: `chat.py` и нажмите `Enter`.

Файл откроется в редакторе. Скопируйте туда следующий код:

```python
import requests

# Адрес чат-API Ollama на вашем компьютере
OLLAMA_URL = "http://localhost:11434/api/chat"

# Имя модели, которую мы скачали
MODEL = "phi3:mini"


def ask_model(messages):
    """Отправляет историю диалога модели и возвращает её ответ."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,          # ждём ответ целиком
        "options": {"temperature": 0.7}
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=300)
    response.raise_for_status()
    # В ответе на /api/chat текст лежит в поле "message" -> "content"
    return response.json()["message"]["content"]


# История диалога: список словарей, по одному на каждую реплику
messages = [
    {"role": "system", "content": "Ты — дружелюбный помощник. Отвечай понятно и по существу."}
]

print("Чат с локальной моделью. Напишите вопрос или 'выход' для завершения.")

# Бесконечный цикл: программа ждёт ввода, пока вы не напишете "выход"
while True:
    user_text = input("\nВы: ")

    if user_text.strip().lower() in ("выход", "exit", "quit"):
        print("До свидания!")
        break

    if user_text.strip() == "":
        continue

    messages.append({"role": "user", "content": user_text})

    print("Модель думает... (первый ответ может занять до минуты)")
    try:
        answer = ask_model(messages)
    except requests.exceptions.ConnectionError:
        print("Ошибка: Ollama не запущена. Проверьте, что программа Ollama работает.")
        messages.pop()
        continue

    messages.append({"role": "assistant", "content": answer})
    print("\nМодель:", answer)
```

Этот код использует эндпоинт `/api/chat`, который хранит историю диалога, поэтому модель «помнит» контекст беседы.

---

## Шаг 9. Установите библиотеку `requests`

Код использует библиотеку `requests` для отправки HTTP-запросов. Её нужно установить один раз.

1. В VS Code откройте терминал: меню **Terminal → New Terminal** (или нажмите `` Ctrl+` ``).
2. Убедитесь, что в правом верхнем углу терминала выбрано **powershell**.
3. Выполните:
   ```powershell
   pip install requests
   ```

---

## Шаг 10. Запустите программу

Есть два равнозначных способа.

**Способ А (через терминал):**
В том же терминале введите:
```powershell
python chat.py
```

**Способ Б (кнопка Play):**
Откройте файл `chat.py` в редакторе и нажмите зелёный треугольник **▷** в правом верхнем углу — VS Code сам запустит файл в терминале.

После запуска в терминале появится:
```
Чат с локальной моделью. Напишите вопрос или 'выход' для завершения.

Вы: _
```
Курсор мигает после `Вы:` — программа ждёт ваш текст. Введите вопрос, нажмите `Enter`, и через некоторое время увидите ответ модели. Чтобы выйти, введите `выход`.

---

## Шаг 11. Если русские буквы превращаются в «кракозябры»

PowerShell иногда неправильно отображает кириллицу. Перед запуском программы выполните в том же терминале три команды:
```powershell
[Console]::OutputEncoding = [System.Text.Encoding]::UTF8
$OutputEncoding = [System.Text.Encoding]::UTF8
chcp 65001 > $null
```
После этого снова запустите `python chat.py`.




## Шаг 1. Скачайте новые модели (Qwen и DeepSeek) через Ollama

Модель `phi3:mini` хороша для старта, но для более качественных ответов и кода стоит попробовать другие варианты. Все команды выполняются в PowerShell.

### Qwen 2.5 (хорошо понимает русский и код)

Qwen 2.5 от Alibaba — сильный универсальный выбор. Для ноутбука с 8–16 ГБ ОЗУ подойдут версии 3B или 7B.

```powershell
# Версия 3B (около 1.9 ГБ) — если мало памяти
ollama pull qwen2.5:3b

# Версия 7B (около 4.7 ГБ) — если 16 ГБ ОЗУ и выше
ollama pull qwen2.5:7b
```

Если задача — именно генерация кода, есть специализированная линейка `qwen2.5-coder`:
```powershell
ollama pull qwen2.5-coder:7b
```

### DeepSeek-R1 (модель с «рассуждением»)

DeepSeek-R1 умеет «думать» перед ответом, что часто даёт более точные результаты в сложных задачах. Для локального запуска доступны облегчённые версии:

```powershell
# Самая лёгкая версия (около 1.1 ГБ)
ollama pull deepseek-r1:1.5b

# Версия 8B (около 5.2 ГБ) — заметно умнее, но требует больше памяти
ollama pull deepseek-r1:8b
```

> **Важно:** тег `deepseek-r1:8b` в каталоге Ollama может указывать на разные версии модели (в частности, на дистиллированную версию Qwen3-8B). Если нужна максимальная предсказуемость, можно использовать явный тег `deepseek-r1:8b-0528-qwen3-q4_K_M`. Для учебных целей достаточно общего `deepseek-r1:8b`.

### DeepSeek-Coder (если нужен именно помощник по коду)

Линейка `deepseek-coder` доступна в размерах 1.3B, 6.7B и 33B. Для ноутбука разумный выбор — 6.7B (около 7 ГБ):
```powershell
ollama pull deepseek-coder:6.7b
```

После скачивания проверьте список:
```powershell
ollama list
```

Чтобы **переключить чат на новую модель**, достаточно изменить одну строку в `chat.py`:
```python
MODEL = "qwen2.5:7b"      # вместо "phi3:mini"
# или
MODEL = "deepseek-r1:8b"
```

---

## Шаг 2. Обновите системный промпт (это тоже часть «улучшения»)

Системный промпт (`role: "system"`) задаёт характер модели. Для психологического чата можно использовать более структурированный вариант, основанный на подходах из практики ИИ-терапии:

```python
messages = [
    {"role": "system", "content": (
        "Ты — бережный и внимательный помощник психолога. "
        "Общайся простым, спокойным языком, без осуждения. "
        "Прежде чем ответить, подумай: (1) какую эмоцию или проблему описывает человек? "
        "(2) что ещё важно уточнить, чтобы понять ситуацию глубже? "
        "Не давай диагнозов и не назначай лечение. "
        "Если человек в остром кризисе, предложи обратиться к специалисту."
    )}
]
```

Этот промпт можно менять под конкретное упражнение (например, «Ты — помощник, который задаёт уточняющие вопросы по технике КПТ»).

---

## Шаг 3. Промпты для DeepSeek / ChatGPT по улучшению кода

Каждый промпт предполагает, что вы **приложите свой текущий `chat.py`** (или его фрагмент). Копируйте промпт, вставляйте в DeepSeek/ChatGPT, добавляйте код и получайте конкретные правки.

### Промпт 1. Сделать ответы «живыми» (стриминг)

**Что делает:** сейчас программа ждёт весь ответ целиком и печатает его одним куском. Стриминг печатает текст по мере генерации — как в ChatGPT. Это заметно улучшает ощущение от чата.

```text
Перепиши мой код chat.py так, чтобы ответ модели выводился в терминал постепенно, по мере генерации (стриминг), а не одним куском в конце.

Используй эндпоинт Ollama /api/chat с параметром stream=True. При обработке ответа учитывай, что Ollama присылает JSON-объекты построчно (каждая строка — отдельный JSON). Печатай только поле message.content каждого чанка, без лишних символов. В конце каждого ответа добавляй перевод строки.

Вот мой текущий код:
[вставьте код chat.py]
```

> **Подсказка для проверки:** в официальной документации Ollama показано, что при `stream=True` ответ приходит по частям, и каждую часть нужно обрабатывать отдельно. Если модель в ответе предлагает `response.json()`, а не построчный разбор — это неверно для стриминга.

---

### Промпт 2. Ограничить историю диалога (чтобы не «съедало» память)

**Что делает:** Ollama загружает всю историю сообщений в память при каждом запросе. В долгом диалоге это может привести к нехватке VRAM/RAM. Решение — обрезать старые сообщения, сохраняя системный промпт и последние N реплик.

```text
Добавь в мой chat.py функцию, которая перед отправкой запроса обрезает историю сообщений, если она становится слишком длинной.

Требования:
1. Всегда сохраняй первое сообщение с ролью "system".
2. Оставляй последние N сообщений (N = 10–15, задай константой в начале файла).
3. Если история обрезается, добавляй в начало (после system) короткое сообщение от assistant с текстом «[Предыдущая часть диалога опущена для экономии памяти]».
4. Функция должна вызываться автоматически перед каждым запросом к модели.

Покажи, как встроить эту функцию в существующий цикл while. Вот мой код:
[вставьте код chat.py]
```

---

### Промпт 3. Обработка ошибок «по-человечески»

**Что делает:** сейчас при ошибке соединения программа просто печатает сообщение и продолжает. Но если Ollama вернула ошибку 404 (модель не найдена) или таймаут — пользователь не понимает, что делать. Промпт просит добавить понятные сообщения и повторные попытки.

```text
Улучши обработку ошибок в моём chat.py. Сейчас там есть только try/except для ConnectionError. Добавь:

1. Отдельную обработку HTTP-ошибок (requests.exceptions.HTTPError): если статус 404 — сообщение «Модель не найдена. Проверьте имя модели и выполните ollama list». Если 500 — «Внутренняя ошибка Ollama. Попробуйте перезапустить Ollama.»
2. Обработку таймаута (requests.exceptions.Timeout): «Модель не ответила за отведённое время. Возможно, не хватает памяти. Попробуйте модель меньше.»
3. При любой ошибке — не завершать программу, а возвращать пользователя в начало цикла (к приглашению "Вы:").
4. Логировать ошибку в файл error.log (кратко: время, тип ошибки, сообщение).

Покажи полный обновлённый код chat.py с этими изменениями. Вот мой текущий код:
[вставьте код chat.py]
```

---

### Промпт 4. Сохранение и загрузка истории диалога

**Что делает:** сейчас при каждом запуске `chat.py` история начинается с нуля. Промпт просит сохранять диалог в JSON-файл и загружать его при следующем запуске — чтобы можно было продолжать беседу.

```text
Добавь в chat.py сохранение истории диалога в файл history.json и загрузку из него при старте.

Требования:
1. Если файл history.json существует — загрузить список messages из него и продолжить с этого места. Если нет — начать с системного промпта, как сейчас.
2. После каждой пары «вопрос пользователя → ответ модели» сохранять обновлённый список messages в history.json (перезаписывать файл).
3. Добавить команду "очистить" (clear): если пользователь вводит её, история сбрасывается к одному системному промпту, а файл history.json перезаписывается.
4. Использовать только стандартную библиотеку json, без внешних зависимостей.

Покажи полный код. Вот мой chat.py:
[вставьте код chat.py]
```

---

### Промпт 5. Добавить «режим психолога» и переключение ролей

**Что делает:** вы можете захотеть, чтобы одна и та же программа работала в разных режимах: «просто помощник», «задающий вопросы по КПТ», «объясняющий термины простыми словами». Промпт просит добавить переключение системного промпта по команде.

```text
Добавь в chat.py возможность переключать «режим» системного промпта командами:
- "режим: помощник" — системный промпт: «Ты — дружелюбный помощник. Отвечай понятно.»
- "режим: кпт" — системный промпт: «Ты — помощник, который задаёт уточняющие вопросы в духе КПТ. Не давай советов, пока не поймёшь ситуацию.»
- "режим: объяснение" — системный промпт: «Ты — терпеливый преподаватель. Объясняй термины простыми словами и с примерами.»

При смене режима история диалога не сбрасывается, но в начало messages подставляется новый system-промпт (старый system заменяется). Текущий режим отображается в приглашении, например: "[кпт] Вы:".

Покажи полный обновлённый код. Вот мой chat.py:
[вставьте код chat.py]
```

---

### Промпт 6. Автоматический подсчёт токенов и предупреждение

**Что делает:** чтобы не «упереться» в лимит контекста модели незаметно, программа может считать примерное количество символов (или токенов) и предупреждать, когда история становится слишком длинной.

```text
Добавь в chat.py подсчёт примерного размера истории в символах (это достаточный прокси для токенов на учебном уровне).

Требования:
1. Перед каждым запросом считай суммарную длину всех content в messages.
2. Если суммарная длина превышает порог (например, 6000 символов) — выводи предупреждение: «История диалога стала длинной. Рекомендую ввести "очистить", чтобы начать заново.»
3. Предупреждение выводится только один раз за сессию (не спамить каждую реплику), пока история не будет очищена.
4. Добавь команду "очистить", которая сбрасывает историю и счётчик предупреждения.

Покажи полный код. Вот мой chat.py:
[вставьте код chat.py]
```

---

### Промпт 7. Улучшить читаемость кода (рефакторинг для новичков)

**Что делает:** просит модель переписать код так, чтобы он был понятнее для студентов, знающих только `for`, `if` и списки — без сложных конструкций, с комментариями на русском.

```text
Перепиши мой chat.py так, чтобы код был максимально понятен студентам-психологам, которые знают только базовый Python (переменные, for, if, списки, словари, функции).

Требования:
1. Никаких генераторов, декораторов, асинхронности, классов и сложных конструкций.
2. Каждая смысловая часть — отдельная функция с понятным названием на русском в комментарии (например, # Функция: отправляет историю модели и возвращает ответ).
3. Внутри функций — короткие комментарии на русском, объясняющие, что происходит на каждом шаге.
4. Сохрани всю текущую логику: цикл while, обработку ошибок, историю сообщений.

Покажи переписанный код и кратко объясни, что изменилось. Вот мой chat.py:
[вставьте код chat.py]
```

---

### Промпт 8. Добавить «горячую» замену модели без правки кода

**Что делает:** сейчас, чтобы сменить модель, нужно редактировать строку `MODEL = "..."`. Промпт просит добавить команду `модель: имя`, которая меняет модель на лету.

```text
Добавь в chat.py команду смены модели без редактирования кода. Если пользователь вводит "модель: qwen2.5:7b", программа:
1. Проверяет, что модель существует (можно через GET http://localhost:11434/api/tags — список доступных моделей).
2. Если модели нет — выводит сообщение «Модель не найдена. Доступные модели: [список]» и не меняет текущую.
3. Если модель есть — обновляет глобальную переменную MODEL и выводит «Модель переключена на ...».
4. История диалога при смене модели сохраняется (не сбрасывается).

Покажи полный код с этой функцией. Вот мой chat.py:
[вставьте код chat.py]
```

---

## Как пользоваться этими промптами (короткая памятка для студентов)

1. **Скачайте модель** (Шаг 1), если хотите попробовать Qwen или DeepSeek вместо `phi3:mini`.
2. **Откройте DeepSeek или ChatGPT**, скопируйте один промпт из списка.
3. **Вставьте в конец промпта свой `chat.py`** (полный текст файла).
4. Получите обновлённый код. **Сравните** его с исходным: что именно изменилось? Прочитайте комментарии.
5. **Запустите** новый код в VS Code (`python chat.py`) и проверьте, что улучшение действительно работает.
6. Если что-то пошло не так — вернитесь к предыдущей версии файла (сохраняйте копии: `chat_v1.py`, `chat_v2.py`).




### Промпт 9. Стриминговый веб-интерфейс на Streamlit (пошагово, для практики)

**Что делает:** просит модель не просто добавить стриминг в терминал, а собрать небольшой веб-интерфейс: поле ввода, история сообщений, кнопка очистки и постепенный вывод ответа. Для новичков это удобнее, чем Flask + JavaScript, потому что весь интерфейс описывается на Python.

```text
Ты — терпеливый преподаватель. Помоги мне пошагово сделать простой стриминговый веб-интерфейс для чата с локальной Ollama.

Не перегружай проект: мне нужен один новый файл app.py, который я смогу запустить рядом с моим chat.py. Мой текущий chat.py прикладываю ниже.

Требования к интерфейсу:
1. Используй Streamlit и requests. Никаких Flask, Django, JavaScript, async, классов и сложных декораторов.
2. В app.py должна быть история сообщений в st.session_state.messages.
3. В начале истории — системный промпт: «Ты — дружелюбный помощник. Отвечай понятно и по существу.»
4. Интерфейс:
   - заголовок «Чат с локальной моделью»;
   - история диалога через st.chat_message("user") и st.chat_message("assistant");
   - поле ввода через st.chat_input("Вы:");
   - кнопка «Очистить историю», которая сбрасывает историю к одному системному промпту.
5. Отправка запроса:
   - POST на http://localhost:11434/api/chat;
   - model = "phi3:mini" (в начале файла константа MODEL, чтобы легко менять);
   - stream = True;
   - options: {"temperature": 0.7}.
6. Стриминг ответа:
   - обрабатывай ответ построчно через response.iter_lines();
   - каждую непустую строку декодируй и разбирай через json.loads;
   - бери поле message.content;
   - отдавай кусочки текста через генератор;
   - выводи генератор в интерфейс через st.write_stream(...), чтобы текст появлялся постепенно.
7. Ошибки:
   - ConnectionError: покажи st.error("Ollama не запущена. Проверьте, что программа Ollama работает.");
   - HTTPError: покажи st.error с кодом и коротким пояснением;
   - Timeout: покажи st.error("Модель не ответила за отведённое время. Попробуйте модель меньше.");
   - после ошибки интерфейс не должен падать.
8. Сохраняй историю так: сначала добавляй сообщение пользователя, затем — ответ ассистента целиком после окончания стриминга.
9. Комментарии в коде — на русском и короткие, для студентов-психологов, которые знают только базовый Python.

Пошаговый формат ответа:
Шаг 1. Кратко объясни план: какие файлы создаём и что устанавливаем.
Шаг 2. Покажи полный код app.py.
Шаг 3. Покажи команды установки и запуска в PowerShell:
   pip install streamlit requests
   streamlit run app.py
Шаг 4. Дай чек-лист проверки: что должно появиться в браузере, как понять, что стриминг действительно работает, что делать, если ответ приходит одним куском.
Шаг 5. Предложи одно простое улучшение на будущее, но не добавляй его в код сразу.

Важно: не меняй мой chat.py, просто создай отдельный app.py. Если для стриминга нужен другой формат разбора ответа Ollama, явно объясни, почему и покажи правильный вариант.

Вот мой текущий chat.py:
[вставьте код chat.py]
```

---

**Как этим пользоваться на практике:**

1. Скопируйте промпт в DeepSeek или ChatGPT.
2. Вставьте в конец полный текст своего `chat.py`.
3. Получите `app.py` и команды запуска.
4. В VS Code создайте новый файл `app.py`, вставьте код.
5. В терминале выполните:
   ```powershell
   pip install streamlit requests
   streamlit run app.py
   ```
6. Откройте ссылку, которую покажет Streamlit (обычно `http://localhost:8501`), и попробуйте задать вопрос. Ответ должен появляться постепенно.

**Что этот промпт тренирует:** понимание разницы между «подождать весь ответ» и «стримить по кусочкам», работу с `iter_lines()` и `json.loads`, а также сборку минимального интерфейса без лишней сложности.